## 把目标建筑所有数据整理出来

In [5]:
import pandas as pd

# 读取CSV文件
df = pd.read_csv('../electricity_source_data/electricity_processed.csv')

# 指定要保留的列
columns_to_keep = [
    'timestamp',
    'Robin_lodging_Renea',
    'Rat_health_Shane',
    'Rat_public_Roma',
    'Wolf_office_Cary',
    'Robin_education_Zenia',
    'Gator_public_Leroy'
]

# 筛选列
df_filtered = df[columns_to_keep]


# 保存为新CSV
df_filtered.to_csv('all_target_data.csv', index=False)

## 稀缺度模拟

In [6]:
import numpy as np
import pandas as pd
from typing import Tuple, List, Dict
import matplotlib.pyplot as plt
import os
import json

class SynchronizedDataScarcityGenerator:
    """
    同步的数据稀缺场景生成器
    确保电力数据和对应天气数据在相同时间点缺失
    保留最后20%的数据作为测试集
    """
    
    def __init__(self, electricity_csv: str, weather_labels_json: str = 'weather_labels.json',
                 weather_dir: str = 'weather_data', timestamp_col: str = 'timestamp',
                 test_ratio: float = 0.2):
        """
        初始化生成器
        
        参数:
            electricity_csv: 电力数据CSV路径
            weather_labels_json: 天气站标签JSON路径
            weather_dir: 天气数据目录
            timestamp_col: 时间戳列名
            test_ratio: 测试集比例（默认0.2，即20%）
        """
        self.timestamp_col = timestamp_col
        self.test_ratio = test_ratio
        
        # 加载电力数据
        print("="*70)
        print("📊 加载电力数据...")
        self.electricity_df = pd.read_csv(electricity_csv)
        self.electricity_df[timestamp_col] = pd.to_datetime(self.electricity_df[timestamp_col])
        self.electricity_df.set_index(timestamp_col, inplace=True)
        
        # 按时间排序
        self.electricity_df = self.electricity_df.sort_index()
        
        self.buildings = [col for col in self.electricity_df.columns]
        self.n_samples = len(self.electricity_df)
        
        # 划分训练集和测试集
        self.train_size = int(self.n_samples * (1 - test_ratio))
        self.train_df = self.electricity_df.iloc[:self.train_size]
        self.test_df = self.electricity_df.iloc[self.train_size:]
        
        print(f"✓ 电力数据: {self.n_samples} 样本, {len(self.buildings)} 个建筑")
        print(f"✓ 时间范围: {self.electricity_df.index.min()} 到 {self.electricity_df.index.max()}")
        print(f"✓ 训练集: {len(self.train_df)} 样本 ({(1-test_ratio)*100:.0f}%)")
        print(f"✓ 测试集: {len(self.test_df)} 样本 ({test_ratio*100:.0f}%)")
        print(f"✓ 训练集时间范围: {self.train_df.index.min()} 到 {self.train_df.index.max()}")
        print(f"✓ 测试集时间范围: {self.test_df.index.min()} 到 {self.test_df.index.max()}")
        
        # 加载天气站映射
        print("\n🗺️  加载建筑-天气站映射...")
        with open(weather_labels_json, 'r') as f:
            weather_labels = json.load(f)
        
        self.building_to_weather = {
            b["building_id"]: b["weather_station"]
            for b in weather_labels["buildings"]
        }
        
        # 统计每个天气站对应的建筑
        self.weather_to_buildings = {}
        for building, station in self.building_to_weather.items():
            if station not in self.weather_to_buildings:
                self.weather_to_buildings[station] = []
            self.weather_to_buildings[station].append(building)
        
        print(f"✓ 映射关系加载完成:")
        for station, buildings in self.weather_to_buildings.items():
            print(f"  • {station}: {len(buildings)} 个建筑")
        
        # 加载所有天气数据
        print("\n🌤️  加载天气数据...")
        self.weather_data = {}
        weather_stations = set(self.building_to_weather.values())
        
        for station in weather_stations:
            # 尝试多种可能的文件名
            possible_files = [
                f"{station}.csv",
                f"{station}_processed.csv"
            ]
            
            weather_file = None
            for fname in possible_files:
                fpath = os.path.join(weather_dir, fname)
                if os.path.exists(fpath):
                    weather_file = fpath
                    break
            
            if weather_file:
                df = pd.read_csv(weather_file)
                df[timestamp_col] = pd.to_datetime(df[timestamp_col])
                df.set_index(timestamp_col, inplace=True)
                
                # 按时间排序
                df = df.sort_index()
                
                # 移除site_id列
                if 'site_id' in df.columns:
                    df = df.drop(columns=['site_id'])
                
                # 划分训练集和测试集
                train_end_time = self.train_df.index.max()
                weather_train = df[df.index <= train_end_time]
                weather_test = df[df.index > train_end_time]
                
                self.weather_data[station] = {
                    'full': df,
                    'train': weather_train,
                    'test': weather_test
                }
                
                print(f"  ✓ {station}: 全部 {df.shape}, 训练 {weather_train.shape}, 测试 {weather_test.shape} - {list(df.columns)[:3]}...")
            else:
                print(f"  ✗ 警告: 找不到天气站 {station} 的数据")
                self.weather_data[station] = None
        
        print("="*70)
    
    def place_distributed_blocks(self, target_block_rate: float, 
                                 block_length_range: Tuple[int, int],
                                 min_gap: int = None) -> Tuple[np.ndarray, List]:
        """放置分散的块"""
        # 只在训练集上应用
        mask = np.zeros(len(self.train_df), dtype=bool)
        target_missing = int(len(self.train_df) * target_block_rate)
        
        min_len, max_len = block_length_range
        avg_len = (min_len + max_len) // 2
        
        if min_gap is None:
            min_gap = avg_len
        
        print(f"   块配置: 长度={min_len}-{max_len}h, 最小间隔={min_gap}h")
        
        placed_blocks = []
        attempts = 0
        max_attempts = 10000
        available_starts = list(range(0, len(self.train_df)))
        
        while np.sum(mask) < target_missing and attempts < max_attempts:
            if not available_starts:
                break
            
            block_len = np.random.randint(min_len, max_len + 1)
            valid_starts = [s for s in available_starts if s + block_len <= len(self.train_df)]
            
            if not valid_starts:
                break
            
            start = np.random.choice(valid_starts)
            end = start + block_len
            
            # 检查距离
            too_close = False
            for existing_block in placed_blocks:
                if not (end + min_gap <= existing_block['start'] or 
                       start >= existing_block['end'] + min_gap):
                    too_close = True
                    break
            
            if not too_close:
                mask[start:end] = True
                placed_blocks.append({'start': start, 'end': end, 'length': block_len})
                available_starts = [s for s in available_starts 
                                   if s >= end + min_gap or s + max_len + min_gap <= start]
            
            attempts += 1
        
        print(f"   放置了 {len(placed_blocks)} 个分散块 (尝试: {attempts})")
        print(f"   块覆盖率: {np.sum(mask)/len(self.train_df)*100:.2f}%")
        
        return mask, placed_blocks
    
    def add_random_points(self, mask: np.ndarray, target_total_rate: float) -> Tuple[np.ndarray, int]:
        """添加随机点缺失"""
        current_missing = np.sum(mask)
        target_total_missing = int(len(self.train_df) * target_total_rate)
        still_needed = target_total_missing - current_missing
        
        if still_needed <= 0:
            return mask, 0
        
        available_idx = np.where(~mask)[0]
        if len(available_idx) == 0:
            return mask, 0
        
        still_needed = min(still_needed, len(available_idx))
        random_idx = np.random.choice(available_idx, size=still_needed, replace=False)
        mask[random_idx] = True
        
        return mask, len(random_idx)
    
    def generate_scenario(self, scenario_type: str, random_seed: int = 42) -> Dict:
        """
        生成指定类型的稀缺场景
        
        参数:
            scenario_type: 'mild', 'heavy', 'extreme'
            random_seed: 随机种子
        
        返回:
            包含所有数据的字典
        """
        np.random.seed(random_seed)
        
        # 配置参数
        configs = {
            'mild': {
                'total_rate': 0.20,
                'block_rate': 0.05,
                'block_range': (12, 12),
                'min_gap': 12,
                'name': '轻度稀缺 (20%)'
            },
            'heavy': {
                'total_rate': 0.40,
                'block_rate': 0.20,
                'block_range': (48, 72),
                'min_gap': 72,
                'name': '重度稀缺 (40%)'
            },
            'extreme': {
                'total_rate': 0.60,
                'block_rate': 0.40,
                'block_range': (120, 240),
                'min_gap': 168,
                'name': '极端稀缺 (60%)'
            }
        }
        
        config = configs[scenario_type]
        
        print(f"\n{'='*70}")
        print(f"🎯 生成 {config['name']} 场景...")
        print(f"   目标: {config['block_rate']*100:.0f}% 块状 + "
              f"{(config['total_rate']-config['block_rate'])*100:.0f}% 随机")
        print(f"   注意: 仅应用于前 {(1-self.test_ratio)*100:.0f}% 的训练数据")
        
        # 1. 生成统一的缺失掩码（基于时间维度）- 只在训练集上应用
        mask, block_info = self.place_distributed_blocks(
            target_block_rate=config['block_rate'],
            block_length_range=config['block_range'],
            min_gap=config['min_gap']
        )
        
        mask, n_random = self.add_random_points(mask, config['total_rate'])
        print(f"   添加了 {n_random} 个随机点")
        
        # 2. 应用缺失到电力数据（所有建筑在相同时间点缺失）
        # 复制训练集和测试集
        train_corrupted = self.train_df.copy()
        test_preserved = self.test_df.copy()  # 测试集保持不变
        
        # 获取缺失的时间戳
        missing_timestamps = train_corrupted.index[mask]
        
        # 在这些时间戳位置设置所有建筑为NaN
        train_corrupted.loc[missing_timestamps, :] = np.nan
        
        # 合并训练集和测试集
        electricity_corrupted = pd.concat([train_corrupted, test_preserved])
        
        print(f"   电力数据: {len(self.buildings)} 个建筑在 {len(missing_timestamps)} 个时间点缺失")
        print(f"   测试集: {len(test_preserved)} 个样本保持完整")
        
        # 3. 应用相同的缺失模式到对应的天气数据
        weather_corrupted = {}
        
        for station, weather_data in self.weather_data.items():
            if weather_data is None:
                weather_corrupted[station] = None
                continue
            
            # 复制训练集和测试集
            train_weather = weather_data['train'].copy()
            test_weather = weather_data['test'].copy()  # 测试集保持不变
            
            # 找到与电力数据时间戳对齐的位置
            common_timestamps = missing_timestamps.intersection(train_weather.index)
            
            # 在相同时间戳位置设置为NaN
            train_weather.loc[common_timestamps, :] = np.nan
            
            # 合并训练集和测试集
            weather_corrupted[station] = pd.concat([train_weather, test_weather])
            
            # 验证缺失率
            weather_missing_count = train_weather.isna().any(axis=1).sum()
            weather_missing_rate = weather_missing_count / len(train_weather) * 100
            affected_buildings = len(self.weather_to_buildings.get(station, []))
            
            print(f"   天气站 {station}: 训练集缺失率 {weather_missing_rate:.2f}% "
                  f"(影响 {affected_buildings} 个建筑)")
        
        # 4. 统计信息
        stats = {
            'n_blocks': len(block_info),
            'block_details': block_info,
            'n_random_points': n_random,
            'total_missing_rate': np.sum(mask) / len(mask) * 100,
            'missing_timestamps': missing_timestamps,
            'train_size': len(self.train_df),
            'test_size': len(self.test_df),
            'train_end_time': self.train_df.index.max(),
            'test_start_time': self.test_df.index.min()
        }
        
        print(f"✓ 训练集实际缺失率: {stats['total_missing_rate']:.2f}%")
        print(f"✓ 整体缺失率: {(np.sum(mask) / self.n_samples) * 100:.2f}%")
        return {
            'electricity': electricity_corrupted,
            'weather': weather_corrupted,
            'mask': mask,
            'stats': stats
        }
    
    def save_scenario(self, scenario_data: Dict, scenario_type: str, 
                     output_dir: str = './scarcity_data'):
        """
        保存场景数据
        
        参数:
            scenario_data: 场景数据字典
            scenario_type: 'mild', 'heavy', 'extreme'
            output_dir: 输出目录
        """
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"\n💾 保存 {scenario_type.upper()} 场景数据...")
        
        # 保存完整电力数据（包含训练集和测试集）
        electricity_path = os.path.join(output_dir, f'data_{scenario_type}_scarcity.csv')
        scenario_data['electricity'].reset_index().to_csv(electricity_path, index=False)
        print(f"  ✓ 完整电力数据: {electricity_path}")
        
        # 分别保存训练集和测试集
        train_end_time = scenario_data['stats']['train_end_time']
        
        # 保存训练集（带缺失）
        train_electricity = scenario_data['electricity'][scenario_data['electricity'].index <= train_end_time]
        train_electricity_path = os.path.join(output_dir, f'data_{scenario_type}_scarcity_train.csv')
        train_electricity.reset_index().to_csv(train_electricity_path, index=False)
        print(f"  ✓ 训练集电力数据: {train_electricity_path}")
        
        # 保存测试集（完整）
        test_electricity = scenario_data['electricity'][scenario_data['electricity'].index > train_end_time]
        test_electricity_path = os.path.join(output_dir, f'data_{scenario_type}_scarcity_test.csv')
        test_electricity.reset_index().to_csv(test_electricity_path, index=False)
        print(f"  ✓ 测试集电力数据: {test_electricity_path}")
        
        # 保存天气数据
        weather_output_dir = os.path.join(output_dir, 'weather')
        os.makedirs(weather_output_dir, exist_ok=True)
        
        weather_paths = []
        weather_train_paths = []
        weather_test_paths = []
        
        for station, weather_df in scenario_data['weather'].items():
            if weather_df is not None:
                # 保存完整天气数据
                weather_path = os.path.join(weather_output_dir, f'{station}_{scenario_type}_scarcity.csv')
                weather_df.reset_index().to_csv(weather_path, index=False)
                weather_paths.append(weather_path)
                
                # 保存训练集天气数据
                weather_train = weather_df[weather_df.index <= train_end_time]
                weather_train_path = os.path.join(weather_output_dir, f'{station}_{scenario_type}_scarcity_train.csv')
                weather_train.reset_index().to_csv(weather_train_path, index=False)
                weather_train_paths.append(weather_train_path)
                
                # 保存测试集天气数据
                weather_test = weather_df[weather_df.index > train_end_time]
                weather_test_path = os.path.join(weather_output_dir, f'{station}_{scenario_type}_scarcity_test.csv')
                weather_test.reset_index().to_csv(weather_test_path, index=False)
                weather_test_paths.append(weather_test_path)
                
                print(f"  ✓ 天气数据: {os.path.basename(weather_path)} (完整/训练/测试)")
        
        # 保存缺失掩码
        mask_path = os.path.join(output_dir, f'mask_{scenario_type}_scarcity.npy')
        np.save(mask_path, scenario_data['mask'])
        print(f"  ✓ 缺失掩码: {mask_path}")
        
        # 保存缺失时间戳列表
        timestamps_path = os.path.join(output_dir, f'missing_timestamps_{scenario_type}.csv')
        pd.DataFrame({
            'timestamp': scenario_data['stats']['missing_timestamps']
        }).to_csv(timestamps_path, index=False)
        print(f"  ✓ 缺失时间戳: {timestamps_path}")
        
        # 保存训练集和测试集划分信息
        split_info_path = os.path.join(output_dir, f'train_test_split_info.json')
        with open(split_info_path, 'w') as f:
            json.dump({
                'train_size': scenario_data['stats']['train_size'],
                'test_size': scenario_data['stats']['test_size'],
                'train_end_time': scenario_data['stats']['train_end_time'].strftime('%Y-%m-%d %H:%M:%S'),
                'test_start_time': scenario_data['stats']['test_start_time'].strftime('%Y-%m-%d %H:%M:%S'),
                'test_ratio': self.test_ratio,
                'total_samples': self.n_samples
            }, f, indent=4)
        print(f"  ✓ 训练测试集划分信息: {split_info_path}")
        
        return {
            'electricity_path': electricity_path,
            'electricity_train_path': train_electricity_path,
            'electricity_test_path': test_electricity_path,
            'weather_dir': weather_output_dir,
            'weather_paths': weather_paths,
            'weather_train_paths': weather_train_paths,
            'weather_test_paths': weather_test_paths,
            'mask_path': mask_path,
            'timestamps_path': timestamps_path,
            'split_info_path': split_info_path
        }
    
    def generate_all_scenarios(self, output_dir: str = './scarcity_data', 
                              random_seed: int = 42):
        """
        生成所有三种场景
        
        参数:
            output_dir: 输出目录
            random_seed: 随机种子
        """
        print("\n" + "="*70)
        print("🚀 同步数据稀缺场景生成器")
        print("="*70)
        print("配置:")
        print(f"  • 数据划分: 前 {(1-self.test_ratio)*100:.0f}% 训练, 后 {self.test_ratio*100:.0f}% 测试")
        print("  • 轻度:  20% = 5% 块(12h) + 15% 随机")
        print("  • 重度:  40% = 20% 块(2-3天) + 20% 随机")
        print("  • 极端:  60% = 40% 块(5-10天) + 20% 随机")
        print("  • ✨ 电力和天气数据在相同时间点缺失")
        print("  • ✨ 稀缺度仅应用于训练集，测试集保持完整")
        print("="*70)
        
        all_scenarios = {}
        
        for scenario_type in ['mild', 'heavy', 'extreme']:
            # 生成场景
            scenario_data = self.generate_scenario(scenario_type, random_seed)
            
            # 保存数据
            paths = self.save_scenario(scenario_data, scenario_type, output_dir)
            
            all_scenarios[scenario_type] = {
                'data': scenario_data,
                'paths': paths
            }
        
        # 生成验证报告
        self.create_verification_report(all_scenarios, output_dir)

        
        return all_scenarios
    
    def create_verification_report(self, all_scenarios: Dict, output_dir: str):
        """
        创建验证报告，确认电力和天气数据的缺失同步
        """
        report_path = os.path.join(output_dir, 'synchronization_verification.txt')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write("电力-天气数据缺失同步验证报告\n")
            f.write("="*80 + "\n")
            f.write(f"生成时间: {pd.Timestamp.now()}\n")
            f.write(f"总样本数: {self.n_samples}\n")
            f.write(f"训练集样本数: {len(self.train_df)} ({(1-self.test_ratio)*100:.0f}%)\n")
            f.write(f"测试集样本数: {len(self.test_df)} ({self.test_ratio*100:.0f}%)\n")
            f.write(f"训练集时间范围: {self.train_df.index.min()} 到 {self.train_df.index.max()}\n")
            f.write(f"测试集时间范围: {self.test_df.index.min()} 到 {self.test_df.index.max()}\n")
            f.write(f"建筑数: {len(self.buildings)}\n")
            f.write("\n建筑-天气站映射:\n")
            for station, buildings in self.weather_to_buildings.items():
                f.write(f"  {station}: {', '.join(buildings[:3])}...")
                if len(buildings) > 3:
                    f.write(f" (共{len(buildings)}个)")
                f.write("\n")
            f.write("\n")
            
            for scenario_type, scenario_info in all_scenarios.items():
                data = scenario_info['data']
                stats = data['stats']
                mask = data['mask']
                
                f.write("-"*80 + "\n")
                f.write(f"{scenario_type.upper()} 场景\n")
                f.write("-"*80 + "\n")
                f.write(f"训练集缺失率: {stats['total_missing_rate']:.2f}%\n")
                f.write(f"整体缺失率: {(np.sum(mask) / self.n_samples) * 100:.2f}%\n")
                f.write(f"缺失样本数: {np.sum(mask)}/{len(self.train_df)} (仅训练集)\n")
                f.write(f"块数量: {stats['n_blocks']}\n")
                f.write(f"随机点数量: {stats['n_random_points']}\n")
                f.write("\n")
                
                # 验证同步
                f.write("同步验证:\n")
                
                # 获取缺失时间戳
                missing_ts = stats['missing_timestamps']
                
                # 检查电力数据
                electricity_missing = data['electricity'].loc[missing_ts].isna().all(axis=1).sum()
                f.write(f"  电力数据在缺失时间点的缺失数: {electricity_missing}/{len(missing_ts)}\n")
                
                # 检查每个天气站
                f.write("\n  天气站同步验证:\n")
                for station, weather_df in data['weather'].items():
                    if weather_df is not None:
                        # 找到共同的时间戳
                        common_ts = missing_ts.intersection(weather_df.index)
                        
                        # 统计在这些时间点的缺失
                        weather_missing = weather_df.loc[common_ts].isna().any(axis=1).sum()
                        sync_rate = weather_missing / len(common_ts) * 100 if len(common_ts) > 0 else 0
                        
                        affected_buildings = self.weather_to_buildings.get(station, [])
                        
                        f.write(f"    {station}:\n")
                        f.write(f"      影响建筑: {len(affected_buildings)} 个\n")
                        f.write(f"      共同时间点: {len(common_ts)}\n")
                        f.write(f"      同步缺失: {weather_missing}/{len(common_ts)}\n")
                        f.write(f"      同步率: {sync_rate:.2f}%\n")
                
                f.write("\n")
            
            f.write("="*80 + "\n")
            f.write("测试集验证:\n")
            f.write("  ✓ 测试集数据保持完整，没有引入缺失\n")
            f.write("="*80 + "\n")
        
        print(f"\n✓ 验证报告已保存: {report_path}")
    


# ==================== 主函数 ====================

def main():
    """主函数"""
    print("\n" + "🌟"*35)
    print("同步数据稀缺场景生成系统")
    print("🌟"*35 + "\n")
    
    # 创建生成器
    generator = SynchronizedDataScarcityGenerator(
        electricity_csv='all_target_data.csv',
        weather_labels_json='weather_labels.json',
        weather_dir='weather_data',
        timestamp_col='timestamp',
        test_ratio=0.2  # 保留最后20%作为测试集
    )
    
    # 生成所有场景
    all_scenarios = generator.generate_all_scenarios(
        output_dir='./scarcity_data',
        random_seed=42
    )
    
    # 打印总结
    print("\n" + "="*70)
    print("✅ 所有场景生成完成!")
    print("="*70)
    
    print("\n📁 生成的文件:")
    print("\n电力数据:")
    for scenario_type in ['mild', 'heavy', 'extreme']:
        print(f"  • 完整数据: {all_scenarios[scenario_type]['paths']['electricity_path']}")
        print(f"  • 训练集: {all_scenarios[scenario_type]['paths']['electricity_train_path']}")
        print(f"  • 测试集: {all_scenarios[scenario_type]['paths']['electricity_test_path']}")
    
    print("\n天气数据目录:")
    print(f"  • ./scarcity_data/weather/")
    for scenario_type in ['mild', 'heavy', 'extreme']:
        n_files = len(all_scenarios[scenario_type]['paths']['weather_paths'])
        print(f"    - {scenario_type}: {n_files} 个完整文件, {n_files} 个训练集文件, {n_files} 个测试集文件")
    
    print("\n缺失掩码:")
    for scenario_type in ['mild', 'heavy', 'extreme']:
        print(f"  • {all_scenarios[scenario_type]['paths']['mask_path']}")
    
    print("\n缺失时间戳:")
    for scenario_type in ['mild', 'heavy', 'extreme']:
        print(f"  • {all_scenarios[scenario_type]['paths']['timestamps_path']}")
    
    print("\n训练测试集划分信息:")
    print(f"  • {all_scenarios['mild']['paths']['split_info_path']}")
    
    print("\n报告和可视化:")
    print("  • ./scarcity_data/synchronization_verification.txt")
    print("  • ./scarcity_data/synchronization_visualization.png")
    
    print("\n" + "="*70)
    print("🎯 关键特性:")
    print(f"  ✓ 数据划分: 前 {(1-generator.test_ratio)*100:.0f}% 训练, 后 {generator.test_ratio*100:.0f}% 测试")
    print("  ✓ 电力和天气数据在相同时间点缺失")
    print("  ✓ 每个天气站影响其对应的所有建筑")
    print("  ✓ 提供完整的验证报告")
    print("  ✓ 精确控制缺失率 (20%, 40%, 60%)")
    print("  ✓ 稀缺度仅应用于训练集，测试集保持完整")
    print("="*70 + "\n")
    
    return all_scenarios, generator

if __name__ == "__main__":
    all_scenarios, generator = main()
        


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
同步数据稀缺场景生成系统
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟

📊 加载电力数据...
✓ 电力数据: 17544 样本, 6 个建筑
✓ 时间范围: 2016-01-01 00:00:00 到 2017-12-31 23:00:00
✓ 训练集: 14035 样本 (80%)
✓ 测试集: 3509 样本 (20%)
✓ 训练集时间范围: 2016-01-01 00:00:00 到 2017-08-07 18:00:00
✓ 测试集时间范围: 2017-08-07 19:00:00 到 2017-12-31 23:00:00

🗺️  加载建筑-天气站映射...
✓ 映射关系加载完成:
  • Hog: 15 个建筑
  • Robin: 5 个建筑
  • Rat: 6 个建筑
  • Eagle: 2 个建筑
  • Wolf: 2 个建筑
  • Gator: 1 个建筑

🌤️  加载天气数据...
  ✓ Wolf: 全部 (17505, 5), 训练 (14001, 5), 测试 (3504, 5) - ['airTemperature', 'dewTemperature', 'seaLvlPressure']...
  ✓ Gator: 全部 (17544, 5), 训练 (14034, 5), 测试 (3510, 5) - ['airTemperature', 'dewTemperature', 'seaLvlPressure']...
  ✓ Rat: 全部 (17539, 5), 训练 (14029, 5), 测试 (3510, 5) - ['airTemperature', 'dewTemperature', 'seaLvlPressure']...
  ✗ 警告: 找不到天气站 Eagle 的数据
  ✗ 警告: 找不到天气站 Hog 的数据
  ✓ Robin: 全部 (17516, 5), 训练 (14011, 5), 测试 (3505, 5) - ['airTemperature', 'dewTemperature', 'seaLvlPressure']...

🚀 同步数据稀缺场景生成器
配置:
  • 数据划分: 前 80